<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Feature Detection and Object Tracking</b></h1>
</div>

## Theoretical Foundations

This notebook formalizes the mathematical and algorithmic basis of the corresponding laboratory implementation. The section order mirrors the experimental workflow so that assumptions, estimation steps, diagnostics, and validation criteria remain directly traceable to the executable notebook.

### Technical Context

The tracker does not learn from a training dataset. Instead, it uses **local visual correspondences** between a fixed reference image and each new video frame.

The central chain is

$$
\boxed{
\text{reference ROI}
\rightarrow
\text{ORB keypoints/descriptors}
\rightarrow
\text{descriptor matches}
\rightarrow
\text{RANSAC inliers}
\rightarrow
H
\rightarrow
\text{transformed box}
}
$$

where $H\in\mathbb{R}^{3\times3}$ is a projective homography.

### Core Feature-Matching Model

A single homography is suitable when the visible object region behaves approximately like a plane, or when camera/object motion produces a transformation close enough to a planar projective mapping.

The tracker also assumes that enough distinctive local texture remains visible so that ORB can find repeatable features.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $\mathbf{x}=[u,v,1]^T$ | homogeneous point in the reference frame |
| $\mathbf{x}'=[u',v',1]^T$ | corresponding point in the current frame |
| $H$ | reference-to-current homography |
| $d_i$ | ORB binary descriptor |
| $D_H(d_i,d_j)$ | Hamming distance between descriptors |
| $m_i$ | one descriptor match |
| $\tau_R$ | RANSAC reprojection threshold |
| $N_m$ | number of descriptor matches |
| $N_i$ | number of RANSAC inliers |
| $r=N_i/N_m$ | inlier ratio |
| $B_0$ | four-corner initial bounding box |
| $B_t$ | transformed bounding box in frame $t$ |

### Analytical Scope

The analysis establishes the following points:

- how ORB detects and describes local visual features;
- why binary descriptors are compared with Hamming distance;
- how descriptor matching produces tentative correspondences;
- why RANSAC is needed before estimating a reliable homography;
- how a homography transforms the initial object box;
- how match count, inlier count, and inlier ratio diagnose tracking quality;
- when a planar projective model is appropriate and where it can fail.

## 1. Validate the Input Video and Output Paths

### Why file validation matters

Tracking is a sequential experiment: every later result depends on reading the same source video correctly.

If the path is wrong, silently switching to another file would destroy reproducibility.

Repository-relative paths such as

$$
\texttt{../data/video1.mp4}
$$

keep the notebook portable inside the module.

### Output evidence

The figures directory is not a cosmetic detail. It stores the visual evidence needed to inspect:

- reference features;
- transformed object boxes;
- RANSAC inlier matches;
- correspondence statistics through time.


## 2. Define the Initial Bounding Box and Tracking Parameters

### Object initialization

The object is specified by

$$
(row,col,height,width)
=
(24,46,170,160).
$$

Image coordinates use column as horizontal coordinate $u$ and row as vertical coordinate $v$.

The four ordered reference corners are therefore

$$
(46,24),
$$

$$
(206,24),
$$

$$
(206,194),
$$

$$
(46,194).
$$

These four points form the polygon $B_0$ that will later be transformed by the estimated homography.

### Why four corners?

A homography maps points in projective image coordinates. Once $H$ is known, transforming only the four box corners is enough to obtain the current quadrilateral.

### Core parameters

The laboratory uses:

$$
N_{features}=1000,
$$

$$
\tau_R=5.0\text{ px},
$$

and at least four matches/inliers.

The theoretical minimum of four point correspondences comes from the eight effective degrees of freedom of a homography.


## 3. Initialize ORB and the Hamming-Distance Matcher

### ORB overview

ORB combines two ideas:

- **FAST-like corner detection** to find repeatable keypoints;
- **BRIEF-like binary descriptors** with orientation handling.

ORB stands for **Oriented FAST and Rotated BRIEF**.

A keypoint stores information such as image location, scale, orientation, and response.

A descriptor summarizes the local appearance around that keypoint.

### Binary descriptors

An ORB descriptor is 256 bits = 32 bytes.

It can be represented as

$$
d\in\{0,1\}^{256}.
$$

### Hamming distance

Because ORB descriptors are binary, Euclidean distance is not the natural comparison.

The Hamming distance counts differing bits:

$$
D_H(d_a,d_b)
=
\sum_{k=1}^{256}
[d_{a,k}\ne d_{b,k}].
$$

Small Hamming distance means the two local patterns are more similar.

### Brute-force matching

For each reference descriptor, BFMatcher searches descriptor candidates in the current frame.

With `crossCheck=True`, a match is retained only when the relation is mutual:

- reference descriptor $a$ chooses current descriptor $b$;
- current descriptor $b$ also chooses reference descriptor $a$.

This is stricter than one-way nearest-neighbor matching.

### Trade-off

Cross-check can remove ambiguous correspondences, but it can also discard potentially useful matches.

The laboratory intentionally uses this simple mutual-matching rule rather than Lowe's ratio test.

## 4. Read and Validate the Reference Frame

### Why the first frame is special

The tracker is **fixed-reference**, not frame-to-frame.

All later frames are matched directly against frame 0.

This avoids accumulation of geometric drift caused by repeatedly chaining transformations:

$$
H_{0\rightarrow t}
$$

is estimated directly rather than as

$$
H_{t-1\rightarrow t}
\cdots
H_{0\rightarrow1}.
$$

### Grayscale conversion

ORB operates on intensity structure, so the BGR image is converted to a single-channel grayscale frame.

Color is not used by this descriptor pipeline.

### ROI validity

The initial box must satisfy

$$
0\le u < W,
$$

$$
0\le v < H_{image}
$$

for all four corners.

An out-of-frame ROI would create an inconsistent reference object definition.


## 5. Detect Reference ORB Features Inside the Object Region

### Why use a mask?

Without a mask, ORB would detect features anywhere in the reference frame, including the background.

The reference mask enforces:

$$
\text{features}
\subseteq
\text{initial object ROI}.
$$

This gives the descriptor set semantic meaning: it represents the object we intend to track.

### Keypoints and descriptors

Suppose ORB detects $N_r$ reference keypoints:

$$
\{
\mathbf{x}_1,
\ldots,
\mathbf{x}_{N_r}
\}.
$$

Each keypoint receives one descriptor:

$$
\{
d_1,
\ldots,
d_{N_r}
\}.
$$

For ORB, the descriptor matrix has shape

$$
N_r\times32
$$

bytes.

### Why more than four features?

Four correspondences are only the theoretical minimum for a homography.

Real images contain:

- descriptor mismatches;
- repeated textures;
- detection noise;
- partial occlusions.

A large feature pool lets RANSAC find a geometrically consistent subset.

### Reference result

The previous execution detected 868 reference keypoints, showing that the object ROI contains substantial local texture.

That historical value is a useful plausibility reference, not a hard-coded acceptance target.

## 6. Visualize the Reference Object and ORB Keypoints

### Why inspect the keypoints?

A numerical count such as "868 keypoints" does not reveal where they are.

A visualization can expose:

- concentration on object texture;
- unexpected detections near ROI boundaries;
- insufficient spatial distribution;
- poor initialization.

### Spatial distribution matters

If all useful correspondences lie in a tiny region, homography estimation becomes less stable than when inliers span a large part of the object plane.

The geometry benefits from points distributed across both image axes.

### Technical Implication

Feature quantity and feature geometry are different concepts. A large number of keypoints does not guarantee a well-conditioned projective estimate.

## 7. Define Frame Matching and RANSAC Homography Estimation

### Step 1 — detect current-frame ORB features

For each current grayscale frame, ORB produces

$$
\{
(\mathbf{x}'_j,d'_j)
\}.
$$

### Step 2 — match descriptors

Reference and current descriptors are matched using Hamming distance.

Each retained match associates

$$
\mathbf{x}_i
\leftrightarrow
\mathbf{x}'_j.
$$

Descriptor similarity alone does not guarantee geometric correctness.

### Step 3 — homography model

A planar homography satisfies

$$
s\mathbf{x}'
=
H\mathbf{x},
$$

with

$$
H=
\begin{bmatrix}
h_{11}&h_{12}&h_{13}\\
h_{21}&h_{22}&h_{23}\\
h_{31}&h_{32}&h_{33}
\end{bmatrix}.
$$

Because overall scale is arbitrary, $H$ has eight effective degrees of freedom.

### From point equations to linear constraints

For

$$
\mathbf{x}=[u,v,1]^T,
$$

and

$$
\mathbf{x}'=[u',v',1]^T,
$$

cross multiplication yields two independent equations per correspondence.

Four non-degenerate matches provide eight equations, which is the minimum needed to estimate $H$ up to scale.

### Why ordinary homography estimation is fragile

If even a few descriptor matches are wrong, least-squares estimation using every match can be strongly corrupted.

This motivates RANSAC.

### RANSAC principle

RANSAC repeatedly:

1. samples a minimal subset;
2. estimates a candidate homography;
3. projects the other matches;
4. measures reprojection error;
5. counts correspondences whose error is below $\tau_R$.

A correspondence is treated as an inlier when its geometric error is consistent with the candidate model.

The laboratory uses

$$
\tau_R=5\text{ px}.
$$

### Inlier mask

RANSAC returns a Boolean decision for each descriptor match:

$$
m_i\in
\{
\text{inlier},
\text{outlier}
\}.
$$

The final homography is supported by the geometrically consistent subset.

### Failure cases

The function rejects a frame when:

- too few current ORB features exist;
- fewer than four matches exist;
- homography estimation fails;
- fewer than four RANSAC inliers remain;
- the returned matrix is non-finite.


## 8. Track the Object Throughout the Video

### Perspective transformation of the box

Once $H_t$ is estimated for frame $t$, each reference corner is transformed:

$$
s_i
\mathbf{b}_{i,t}
=
H_t
\mathbf{b}_{i,0}.
$$

After homogeneous division:

$$
u'
=
\frac{x'}{w'},
\qquad
v'
=
\frac{y'}{w'}.
$$

Applying this to all four corners produces $B_t$.

### Why the tracked box is a quadrilateral

Under perspective projection, an axis-aligned rectangle generally does not remain axis-aligned.

Therefore the correct visualization is the transformed four-corner polygon, not a new horizontal rectangle.

### Frame rejection

If the homography or transformed coordinates are invalid, the frame is marked as failed.

This is preferable to drawing a geometrically meaningless box.

### Memory-aware design

The refactored notebook stores only compact per-frame data:

- frame index;
- success/failure;
- failure reason;
- match/inlier counts;
- $H_t$;
- transformed box.

It does **not** store every decoded video frame.

Selected images are re-read only when figures are produced.

### Why this matters

For hundreds of frames, keeping multiple BGR image copies can consume hundreds of megabytes with no analytical benefit.

## 9. Compute Tracking Summary Metrics

### Match count

For frame $t$:

$$
N_m(t)
=
\text{number of mutual descriptor matches}.
$$

This measures descriptor-level correspondence availability.

### Inlier count

$$
N_i(t)
=
\text{number of matches accepted by RANSAC}.
$$

This measures how many correspondences support one coherent homography.

### Inlier ratio

$$
r(t)
=
\frac{N_i(t)}{N_m(t)}.
$$

By definition,

$$
0\le r(t)\le1.
$$

### Interpretation

A high $N_m$ with low $r$ means:

> many local appearances matched, but relatively few agree geometrically.

A smaller $N_m$ with high $r$ can indicate:

> fewer matches, but a geometrically cleaner set.

Therefore match count and inlier ratio should be interpreted together.

### Tracking success rate

If $N_s$ of $N_p$ processed frames produce accepted geometry:

$$
S
=
\frac{N_s}{N_p}.
$$

The previous execution produced 850 successful frames and no failures. That value is a historical reference for this specific video/configuration, not a universal requirement.

### Important caution

Inlier ratio is a proxy for correspondence consistency, not direct ground-truth localization accuracy.

Without manually annotated object positions, the notebook cannot compute true tracking error in pixels or IoU.

## 10. Visualize Representative Tracking Frames

### Why sample across time?

Inspecting only an early frame can hide later failures.

The notebook selects several successful frames spread across the sequence.

This gives a qualitative view of:

- perspective change;
- object motion;
- box stability;
- difficult portions of the video.

### Re-reading frames

Because only compact results are stored, the video is seeked to selected indices using

$$
\texttt{CAP\_PROP\_POS\_FRAMES}.
$$

This trades a small amount of I/O for significantly lower memory usage.

### Evaluation limitation

A visually plausible box is not quantitative ground truth.

It should be combined with correspondence diagnostics rather than used as the only evidence.

## 11. Visualize RANSAC Inlier Matches

### Why visualize only inliers?

A raw descriptor-match figure can be dominated by wrong correspondences.

Using the RANSAC mask shows the subset that supports the estimated projective model.

### Geometric pattern

For a good frame, inlier lines should generally connect consistent regions of the object between the reference and current image.

Strongly crossing or spatially incoherent lines may suggest:

- repeated texture;
- descriptor ambiguity;
- wrong geometry;
- insufficient model validity.

### Representative frame

The notebook uses a temporal middle successful result and recomputes its full features/matches solely for visualization.

This avoids storing bulky keypoint and match objects for every frame.


## 12. Analyze Matches, Inliers, and Inlier Ratio Across the Sequence

### Temporal diagnostic 1 — counts

Plot

$$
N_m(t)
$$

and

$$
N_i(t)
$$

against frame index.

This reveals where the feature-matching problem becomes easier or harder.

### Temporal diagnostic 2 — ratio

Plot

$$
r(t)
=
\frac{N_i(t)}{N_m(t)}.
$$

Sudden ratio drops can indicate:

- motion blur;
- occlusion;
- strong viewpoint change;
- appearance change;
- repetitive background features.

### Example interpretation

If the ratio drops but the transformed box still looks plausible, the homography may still be supported by enough inliers.

If both inlier count and ratio collapse, confidence in the geometry should decrease.

### Why no universal threshold?

A ratio such as 0.5 is not inherently "good" or "bad" across all videos.

Interpretation depends on:

- number and distribution of inliers;
- object texture;
- motion;
- imaging noise;
- downstream tolerance.

### Technical Implication

Tracking diagnostics are strongest when **counts, ratios, and visual geometry agree**.

## 13. Run Numerical and Output-file Validation Checks

### Descriptor validation

The reference descriptor matrix must remain consistent with ORB:

$$
N_r\times32.
$$

### Frame accounting

If the video contains $N$ decoded frames and frame 0 is the reference, the tracker should attempt

$$
N-1
$$

current frames.

### Homography validation

Every accepted result must satisfy:

$$
H_t\in\mathbb{R}^{3\times3}
$$

with finite entries.

The transformed box must satisfy:

$$
B_t\in\mathbb{R}^{4\times1\times2}
$$

with finite coordinates.

### Correspondence logic

For a successful frame:

$$
N_m\ge4,
$$

$$
N_i\ge4,
$$

and

$$
N_i\le N_m.
$$

Therefore

$$
0\le\frac{N_i}{N_m}\le1.
$$

### Output validation

The five expected diagnostics are:

1. reference ORB keypoints;
2. representative tracking frames;
3. RANSAC inlier matches;
4. matches/inliers by frame;
5. inlier ratio by frame.

### Validation Interpretation

A trustworthy tracking result is not established by one green polygon. It requires agreement between feature evidence, robust geometric consistency, plausible transformed geometry, and reproducible diagnostic outputs.

## Technical Synthesis

The tracking formulation links local appearance and projective geometry through a single auditable chain:

$$
\boxed{
\text{Reference ROI}
\rightarrow
\text{ORB keypoints/descriptors}
\rightarrow
\text{descriptor matches}
\rightarrow
\text{RANSAC inliers}
\rightarrow
H
\rightarrow
\text{projected object support}
\rightarrow
\text{temporal diagnostics}
}
$$

Descriptor similarity alone is insufficient for reliable tracking; the homography and its RANSAC inlier set provide the geometric consistency test. Match count, inlier count, inlier ratio, projected support, and representative inlier visualizations form the principal evidence used to assess sequence-level behavior.

## Scope and Limitations

### Included

- manual ROI initialization;
- ORB feature detection/description;
- Hamming-distance matching;
- BFMatcher cross-check;
- RANSAC homography;
- perspective-transformed object polygon;
- per-frame match/inlier diagnostics;
- explicit validity checks.

### Not included

- optical flow;
- learned descriptors;
- CNN/Transformer tracking;
- temporal motion prediction;
- Kalman filtering;
- Lowe ratio test;
- multi-object tracking;
- non-rigid deformation models;
- ground-truth IoU or localization-error evaluation.

The method is therefore best interpreted as a rigorous study of **local features + robust projective geometry for tracking**, rather than a general-purpose modern tracker.